This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations as gx
import logging

In [ ]:
from great_expectations_experimental.expectations.expect_queried_custom_query_to_return_num_rows import ExpectQueriedCustomQueryToReturnNumRows

In [2]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [3]:
context = gx.get_context(context_root_dir=gx_context_root_dir)
context.list_expectation_suites()

[ExpectationSuiteIdentifier::gfw-google-827.constraints.segs_activity.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segs_activity.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.encounters.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.features_.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.features_.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.ssvids_identities.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.ssvids_identities.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.fishing_score_.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.fishing_score_.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.segs_activity.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segs_activity.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.messages.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.messages.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.seg

In [4]:
import yaml

In [5]:
from datetime import date,datetime

In [6]:
logging.basicConfig(level=logging.DEBUG, force = True)

In [7]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [8]:
datasource_config.get("project")

'gfw-google-827'

In [9]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [10]:
gx_datasource.get_asset_names()

{'encounters-2.5',
 'encounters-3.0.0',
 'features_-2.5',
 'features_-3.0.0',
 'fishing_score_-2.5',
 'fishing_score_-3.0.0',
 'fragments-3.0.0',
 'loitering-2.5',
 'loitering-3.0.0',
 'messages-2.5',
 'messages-3.0.0',
 'messages_positions-2.5',
 'messages_positions-3.0.0',
 'messages_scored_-2.5',
 'messages_scored_-3.0.0',
 'messages_segmented_-2.5',
 'messages_segmented_-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_identity_daily_-2.5',
 'segment_identity_daily_-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segment_vessel_daily_-2.5',
 'segment_vessel_daily_-3.0.0',
 'segments-2.5',
 'segments-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 've

In [11]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.encounters.2-5',
 'gfw-google-827.alerts.encounters.3-0-0',
 'gfw-google-827.alerts.features_.2-5',
 'gfw-google-827.alerts.features_.3-0-0',
 'gfw-google-827.alerts.fishing_score_.2-5',
 'gfw-google-827.alerts.fishing_score_.3-0-0',
 'gfw-google-827.alerts.fragments.3-0-0',
 'gfw-google-827.alerts.loitering.2-5',
 'gfw-google-827.alerts.loitering.3-0-0',
 'gfw-google-827.alerts.messages.2-5',
 'gfw-google-827.alerts.messages.3-0-0',
 'gfw-google-827.alerts.messages_positions.2-5',
 'gfw-google-827.alerts.messages_positions.3-0-0',
 'gfw-google-827.alerts.messages_scored_.2-5',
 'gfw-google-827.alerts.messages_scored_.3-0-0',
 'gfw-google-827.alerts.messages_segmented_.2-5',
 'gfw-google-827.alerts.messages_segmented_.3-0-0',
 'gfw-google-827.alerts.satellite_timing_offsets.2-5',
 'gfw-google-827.alerts.satellite_timing_offsets.3-0-0',
 'gfw-google-827.alerts.segment_identity_daily_.2-5',
 'gfw-google-827.alerts.segment_identity_daily_.3-0-0',
 'gfw-google-827.a

In [12]:
TABLE_NAME = "messages_segmented_.3-0-0"
DUMMY_BATCH_DATE = '2013-02-01' # the date slice you want to run interactive expectations on

In [14]:
for current_expectation_suite_name in [es for es in context.list_expectation_suite_names() if TABLE_NAME in es and 'constraints' in es]:
    print(current_expectation_suite_name)
    current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
    current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        br_options={DATE_PARTITION_COLUMN: DUMMY_BATCH_DATE}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)
    gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)
    gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)


    gx_validator.expect_column_values_to_be_unique('msgid')
    gx_validator.expect_column_values_to_not_be_null('msgid')

    def timestamp_from_id_sql(id_column: str):
            return(f"""
        ((
            SELECT 
            TIMESTAMP(STRING_AGG(arr, '-'))
            FROM UNNEST(SPLIT({id_column}, '-')) AS arr WITH OFFSET as offset
            WHERE offset BETWEEN 1 and 3
        ))""")
    
    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
            SELECT frag_id
            FROM {{active_batch}}
            WHERE frag_id IS NOT NULL
            AND DATE({timestamp_from_id_sql('frag_id')}) != DATE(timestamp)
        """}, value=0, meta={
                    "notes": {
                        "format": "markdown",
                        "content": "All timestamps in a frag_id should be on the same date as timestamp if frag_id is not null.",
                }
        })
    
    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE frag_id IS NOT NULL
        AND LEFT(frag_id, STRPOS(frag_id, "-")-1) != ssvid
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "The `frag_id` should start with the `ssvid`.",
            }
    })
    
    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE frag_id IS NOT NULL
        AND seg_id IS NOT NULL                                                                           
        AND DATE({timestamp_from_id_sql('frag_id')}) < DATE({timestamp_from_id_sql('seg_id')})
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "All timestamps in a seg_id should be on or before `frag_id`'s timestamp."
            }
    })

    gx_validator.expect_column_distinct_values_to_be_in_set("source", ["spire", "orbcomm", "ais-listener", "exactearth"])
    # regex word start in BQ: https://stackoverflow.com/a/60728787/4166885
    gx_validator.expect_column_values_to_match_regex("type", "(?:^|\s)AIS.*")
    gx_validator.expect_column_distinct_values_to_be_in_set("receiver_type", ["satellite", "terrestrial", "dynamic"])

    gx_validator.expect_column_values_to_be_between(
        "lon", 
        min_value = -181, 
        max_value=181, 
        strict_min=True,
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("lon").notnull()'
    )

    gx_validator.expect_column_values_to_be_between(
        "lat", 
        min_value = -91, 
        max_value=91, 
        strict_min=True,
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("lat").notnull()'
    )

    gx_validator.expect_column_values_to_be_between(
        "course", 
        min_value = 0, 
        max_value=360, 
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("course").notnull()'
    )

    gx_validator.expect_column_values_to_be_between(
        "heading", 
        min_value = 0, 
        max_value=360, 
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("heading").notnull()'
    )

    gx_validator.expect_column_values_to_be_between(
        "speed", 
        min_value = 0, 
        max_value=102.3, 
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("speed").notnull()'
    )

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE speed IS NOT NULL
        AND type = "AIS.27"
        AND
        (
            speed < 0
        OR
            speed >= 63
        )
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "Speed should be between 0 and 63 if type is AIS.27",
            }
    })

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE type IN ('AIS.5', 'AIS.19', 'AIS.21', 'AIS.24')
        AND (shipname = '@@@@@@@@@@@@@@@@@@@@')
    """}, value=0)

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE type IN ('AIS.5', 'AIS.24')
        AND shipname = '@@@@@@@'
    """}, value=0)

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE type IN ('AIS.5')
        AND destination = '@@@@@@@@@@@@@@@@@@@@'
    """}, value=0)
    
    # if some expectations fail "as expected" leaves this False
    # setting discard_failed_expectations=True could be useful when experimenting a lot
    gx_validator.save_expectation_suite(discard_failed_expectations=False)



  gx_validator.expect_column_values_to_match_regex("type", "(?:^|\s)AIS.*")

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values


gfw-google-827.constraints.messages_segmented_.3-0-0


DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/abac34fc-5f5a-4b31-807c-f17ecbb9df4f?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:google.api_core.retry:Retrying due to , sleeping 0.7s ...
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/abac34fc-5f5a-4b31-807c-f17ecbb9df4f?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:google.api_core.retry:Retrying due to , sleeping 0.2s ...
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/abac34fc-5f5a-4b31-807c-f17ecbb9df4f?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:google.api_core.retry:Retrying due to , sleeping 1.6s ...
DEBU

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_b79b5836?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_b79b5836
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/7bc0055d-7428-42db-b99e-6116761066d3?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 3f19ea772d4a1a9a2157a7a73f0ce7bb
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

...

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/997de6aa-6180-4c42-ab71-9aea78755799?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.core.expectation_configuration:evaluation_parameters have already been built on this expectation


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/eb4d11ff-8bce-44da-89c7-f032ea36af80?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
INFO:great_expectations.validator.validator:	17 expectation(s) included in expectation_suite.
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): stats.greatexpectations.io:443


DEBUG:urllib3.connectionpool:https://stats.greatexpectations.io:443 "POST /great_expectations/v1/usage_statistics HTTP/1.1" 201 18
DEBUG:great_expectations.core.usage_statistics.usage_statistics:Posted usage stats: message status 201


In [15]:
gx_validator.get_expectation_suite().expectations

INFO:great_expectations.validator.validator:	17 expectation(s) included in expectation_suite.


[{"kwargs": {"column": "msgid"}, "meta": {}, "expectation_type": "expect_column_values_to_be_unique"},
 {"kwargs": {"column": "msgid"}, "meta": {}, "expectation_type": "expect_column_values_to_not_be_null"},
 {"kwargs": {"template_dict": {"user_query": "\n            SELECT frag_id\n            FROM {active_batch}\n            WHERE frag_id IS NOT NULL\n            AND DATE(\n        ((\n            SELECT \n            TIMESTAMP(STRING_AGG(arr, '-'))\n            FROM UNNEST(SPLIT(frag_id, '-')) AS arr WITH OFFSET as offset\n            WHERE offset BETWEEN 1 and 3\n        ))) != DATE(timestamp)\n        "}, "value": 0}, "meta": {"notes": {"format": "markdown", "content": "All timestamps in a frag_id should be on the same date as timestamp if frag_id is not null."}}, "expectation_type": "expect_queried_custom_query_to_return_num_rows"},
 {"kwargs": {"template_dict": {"user_query": "\n        SELECT *\n        FROM {active_batch}\n        WHERE frag_id IS NOT NULL\n        AND LEFT(fr